# M1 Event Type One-vs-Normal Gate 검증

## tl;dr

`normal vs event` 단일 gate가 불안정했기 때문에, 이번 노트북은 `fault vs normal`, `task vs normal`, `activity vs normal`을 분리해 검증한다.

## Context & Methods

- 입력은 16번 M1 taxonomy/window audit 산출물을 사용한다.
- normal 35건은 변경하지 않는다.
- target class별 policy variant를 비교한다.
- compact13과 expanded154, Logistic/RandomForest/ExtraTrees를 비교한다.
- 품질 검증은 생성 CSV를 다시 읽어 독립 재계산한다.

In [1]:

from __future__ import annotations

from pathlib import Path
import subprocess
import warnings

import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().resolve()
OUTPUT_DIR = PROJECT_ROOT / '07_데이터산출물'
SOURCE_DIR = PROJECT_ROOT / '05_데이터셋' / 'PreDist'
ZIP_PATH = SOURCE_DIR / 'predist_dataset.zip'

WINDOW_AUDIT_PATH = OUTPUT_DIR / 'm1_4class_window_policy_audit.csv'
FEATURE_SET_PATH = OUTPUT_DIR / 'm1_compact_feature_set_summary.csv'

OUT_DATASET_SUMMARY = OUTPUT_DIR / 'm1_event_type_one_vs_normal_dataset_summary.csv'
OUT_CV_METRICS = OUTPUT_DIR / 'm1_event_type_one_vs_normal_cv_metrics.csv'
OUT_PREDICTIONS = OUTPUT_DIR / 'm1_event_type_one_vs_normal_predictions.csv'
OUT_THRESHOLD_METRICS = OUTPUT_DIR / 'm1_event_type_one_vs_normal_threshold_metrics.csv'
OUT_CLASS_METRICS = OUTPUT_DIR / 'm1_event_type_one_vs_normal_class_metrics.csv'
OUT_CONFUSION = OUTPUT_DIR / 'm1_event_type_one_vs_normal_confusion_matrix.csv'
OUT_ERROR_AUDIT = OUTPUT_DIR / 'm1_event_type_one_vs_normal_error_audit.csv'
OUT_DECISION_MATRIX = OUTPUT_DIR / 'm1_event_type_one_vs_normal_decision_matrix.csv'
OUT_QUALITY_AUDIT = OUTPUT_DIR / 'm1_event_type_one_vs_normal_quality_audit.csv'
OUT_SPECIAL_EVENT_AUDIT = OUTPUT_DIR / 'm1_event_type_one_vs_normal_special_event_audit.csv'
OUT_REPORT = OUTPUT_DIR / '19_M1_event_type_one_vs_normal_gate_validation_보고서.md'

RANDOM_STATE = 42
N_SPLITS = 5
THRESHOLDS = [0.4, 0.5, 0.6]
DEFAULT_THRESHOLD = 0.5
TARGET_CLASSES = ['fault', 'task', 'activity']

assert WINDOW_AUDIT_PATH.exists(), WINDOW_AUDIT_PATH
assert FEATURE_SET_PATH.exists(), FEATURE_SET_PATH
print('project_root:', PROJECT_ROOT)
print('output_dir:', OUTPUT_DIR)


project_root: C:\3rd_Project\HeatGridAgent
output_dir: C:\3rd_Project\HeatGridAgent\07_데이터산출물


## Data

In [2]:

window_audit = pd.read_csv(WINDOW_AUDIT_PATH)
feature_sets = pd.read_csv(FEATURE_SET_PATH)

print('window_audit', window_audit.shape)
print('feature_sets')
print(feature_sets[['feature_set', 'feature_count']].to_string(index=False))


window_audit (298, 190)
feature_sets
      feature_set  feature_count
           base70             70
      expanded154            154
compact13_overlap             13
   compact20_main             20
  compact27_union             27


In [3]:

def as_bool(series: pd.Series) -> pd.Series:
    if series.dtype == bool:
        return series.fillna(False)
    return series.astype(str).str.lower().isin(['true', '1', 'yes'])


def overlap_count(frame: pd.DataFrame) -> pd.Series:
    return pd.to_numeric(frame['overlap_disturbance_count'], errors='coerce').fillna(0)


def parse_feature_list(feature_set_name: str) -> list[str]:
    row = feature_sets.loc[feature_sets['feature_set'].eq(feature_set_name)]
    assert len(row) == 1, feature_set_name
    features = [feature for feature in str(row.iloc[0]['features']).split('|') if feature]
    missing = sorted(set(features) - set(window_audit.columns))
    assert not missing, f'{feature_set_name} missing features: {missing[:10]}'
    return features


FEATURE_COLUMNS = {
    'compact13': parse_feature_list('compact13_overlap'),
    'expanded154': parse_feature_list('expanded154'),
}
assert len(FEATURE_COLUMNS['compact13']) == 13
assert len(FEATURE_COLUMNS['expanded154']) == 154
print({k: len(v) for k, v in FEATURE_COLUMNS.items()})


{'compact13': 13, 'expanded154': 154}


## Build One-vs-Normal Datasets

In [4]:

def pick_event_rows(class_name: str, policy_variant: str) -> pd.DataFrame:
    eligible = as_bool(window_audit['coverage_eligible'])
    base = window_audit.loc[window_audit['final_class'].eq(class_name) & eligible].copy()
    if class_name == 'fault':
        policies = ['fault_pre_7d']
        require_no_overlap = policy_variant == 'no_overlap'
    elif class_name == 'task':
        if policy_variant == 'pre_all':
            policies = ['task_pre_7d']; require_no_overlap = False
        elif policy_variant == 'post_all':
            policies = ['task_post_7d']; require_no_overlap = False
        elif policy_variant == 'no_overlap':
            policies = ['task_post_7d', 'task_pre_7d']; require_no_overlap = True
        else:
            raise ValueError(policy_variant)
    elif class_name == 'activity':
        if policy_variant == 'pre_all':
            policies = ['activity_pre_7d']; require_no_overlap = False
        elif policy_variant == 'post_all':
            policies = ['activity_post_7d']; require_no_overlap = False
        elif policy_variant == 'no_overlap':
            policies = ['activity_pre_7d', 'activity_post_7d']; require_no_overlap = True
        else:
            raise ValueError(policy_variant)
    else:
        raise ValueError(class_name)

    if require_no_overlap:
        base = base.loc[overlap_count(base).eq(0)].copy()
    rank = {policy: i for i, policy in enumerate(policies)}
    base['policy_rank'] = base['window_policy'].map(rank)
    base = base.loc[base['policy_rank'].notna()].copy()
    base = base.sort_values(['disturbance_row_id', 'policy_rank', 'coverage_rate'], ascending=[True, True, False])
    return base.drop_duplicates(subset=['disturbance_row_id'], keep='first').drop(columns=['policy_rank'])


def normal_rows() -> pd.DataFrame:
    normal = window_audit.loc[
        window_audit['final_class'].eq('normal')
        & window_audit['window_policy'].eq('normal_event_7d')
        & as_bool(window_audit['coverage_eligible'])
    ].copy()
    assert len(normal) == 35
    return normal


def finalize_dataset(target_class: str, policy_variant: str, event_rows: pd.DataFrame) -> pd.DataFrame:
    normal = normal_rows()
    data = pd.concat([normal, event_rows], ignore_index=True).copy()
    dataset = f'{target_class}_{policy_variant}'
    data['dataset'] = dataset
    data['target_class'] = target_class
    data['policy_variant'] = policy_variant
    data['binary_label'] = np.where(data['final_class'].eq(target_class), 1, 0)
    data['binary_label_name'] = np.where(data['binary_label'].eq(1), target_class, 'normal')
    data['substation_id'] = data['substation_id'].astype(int)
    assert data['binary_label'].nunique() == 2, dataset
    assert not data['source_id'].duplicated().any(), dataset
    return data


DATASETS = {}
for target in TARGET_CLASSES:
    variants = ['no_overlap'] if target == 'fault' else ['pre_all', 'post_all', 'no_overlap']
    if target == 'fault':
        variants = ['pre_all', 'no_overlap']
    for variant in variants:
        event_rows = pick_event_rows(target, variant)
        DATASETS[f'{target}_{variant}'] = finalize_dataset(target, variant, event_rows)

for name, data in DATASETS.items():
    print('\n==', name, data.shape)
    print(data.groupby(['binary_label_name', 'final_class', 'window_policy']).size().to_string())
    print('event_overlap_rows:', int(((data['binary_label'] == 1) & (overlap_count(data) > 0)).sum()))



== fault_pre_all (97, 195)
binary_label_name  final_class  window_policy  
fault              fault        fault_pre_7d       62
normal             normal       normal_event_7d    35
event_overlap_rows: 7

== fault_no_overlap (90, 195)
binary_label_name  final_class  window_policy  
fault              fault        fault_pre_7d       55
normal             normal       normal_event_7d    35
event_overlap_rows: 0

== task_pre_all (72, 195)
binary_label_name  final_class  window_policy  
normal             normal       normal_event_7d    35
task               task         task_pre_7d        37
event_overlap_rows: 27

== task_post_all (76, 195)
binary_label_name  final_class  window_policy  
normal             normal       normal_event_7d    35
task               task         task_post_7d       41
event_overlap_rows: 4

== task_no_overlap (72, 195)
binary_label_name  final_class  window_policy  
normal             normal       normal_event_7d    35
task               task         task_post

In [5]:

summary_rows = []
for dataset_name, data in DATASETS.items():
    event = data.loc[data['binary_label'].eq(1)]
    summary_rows.append({
        'dataset': dataset_name,
        'target_class': data['target_class'].iloc[0],
        'policy_variant': data['policy_variant'].iloc[0],
        'rows': len(data),
        'normal_rows': int((data['binary_label'] == 0).sum()),
        'target_rows': int((data['binary_label'] == 1).sum()),
        'substation_count': int(data['substation_id'].nunique()),
        'target_substation_count': int(event['substation_id'].nunique()),
        'target_overlap_rows': int((overlap_count(event) > 0).sum()),
        'coverage_min': float(data['coverage_rate'].min()),
        'coverage_median': float(data['coverage_rate'].median()),
        'window_policies': '|'.join(sorted(event['window_policy'].dropna().unique())),
    })
dataset_summary = pd.DataFrame(summary_rows)
dataset_summary.to_csv(OUT_DATASET_SUMMARY, index=False, encoding='utf-8-sig')

special_rows = []
for event_id, flag_col, note in [
    (20, 'event20_low_coverage_flag', 'fault low coverage flag'),
    (34, 'event34_unknown_flag', 'fault unknown label flag'),
    (69, 'event69_unknown_training_end_missing_flag', 'fault unknown label and missing training end flag'),
    (67, 'event67_long_anomaly_flag', 'fault long anomaly flag'),
]:
    sub = window_audit.loc[(window_audit['final_class'] == 'fault') & (pd.to_numeric(window_audit['fault_event_id'], errors='coerce') == event_id)].copy()
    special_rows.append({
        'event_id': event_id,
        'scope': 'fault_metadata',
        'present_in_audit': bool(len(sub) > 0),
        'flag_column': flag_col,
        'flag_value': bool(as_bool(sub[flag_col]).any()) if len(sub) and flag_col in sub else False,
        'treatment': note,
        'report_policy': 'kept as audit metadata; not silently relabeled',
    })
for event_id in [19, 68, 35, 48]:
    sub = window_audit.loc[(window_audit['final_class'] == 'normal') & (pd.to_numeric(window_audit['source_event_id'], errors='coerce') == event_id)].copy()
    special_rows.append({
        'event_id': event_id,
        'scope': 'normal_hard_metadata',
        'present_in_audit': bool(len(sub) > 0),
        'flag_column': 'hard_normal_tag',
        'flag_value': str(sub['hard_normal_tag'].dropna().iloc[0]) if len(sub) and sub['hard_normal_tag'].notna().any() else '',
        'treatment': 'normal label kept with hard/review metadata',
        'report_policy': 'normal label unchanged; tag retained for interpretation',
    })
special_audit = pd.DataFrame(special_rows)
special_audit.to_csv(OUT_SPECIAL_EVENT_AUDIT, index=False, encoding='utf-8-sig')

dataset_summary


,dataset,target_class,policy_variant,rows,normal_rows,target_rows,substation_count,target_substation_count,target_overlap_rows,coverage_min,coverage_median,window_policies
0,fault_pre_all,fault,pre_all,97,35,62,32,27,7,0.995040,1.0,fault_pre_7d
1,fault_no_overlap,fault,no_overlap,90,35,55,31,26,0,0.995040,1.0,fault_pre_7d
2,task_pre_all,task,pre_all,72,35,37,30,21,27,1.000000,1.0,task_pre_7d
3,task_post_all,task,post_all,76,35,41,31,22,4,0.982143,1.0,task_post_7d
4,task_no_overlap,task,no_overlap,72,35,37,31,22,0,0.982143,1.0,task_post_7d
5,activity_pre_all,activity,pre_all,82,35,47,27,19,4,0.995040,1.0,activity_pre_7d
6,activity_post_all,activity,post_all,81,35,46,27,19,5,0.995040,1.0,activity_post_7d
7,activity_no_overlap,activity,no_overlap,81,35,46,27,19,0,0.995040,1.0,activity_post_7d|activity_pre_7d


## Models

In [6]:

MODEL_NAMES = ['dummy_most_frequent', 'logistic_balanced', 'random_forest_balanced_depth3', 'extra_trees_balanced_depth3']


def build_model(model_name: str):
    if model_name == 'dummy_most_frequent':
        return DummyClassifier(strategy='most_frequent')
    if model_name == 'logistic_balanced':
        return Pipeline([
            ('imputer', SimpleImputer(strategy='median', keep_empty_features=True)),
            ('scaler', StandardScaler()),
            ('model', LogisticRegression(class_weight='balanced', max_iter=5000, solver='liblinear', random_state=RANDOM_STATE)),
        ])
    if model_name == 'random_forest_balanced_depth3':
        return Pipeline([
            ('imputer', SimpleImputer(strategy='median', keep_empty_features=True)),
            ('model', RandomForestClassifier(n_estimators=300, max_depth=3, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)),
        ])
    if model_name == 'extra_trees_balanced_depth3':
        return Pipeline([
            ('imputer', SimpleImputer(strategy='median', keep_empty_features=True)),
            ('model', ExtraTreesClassifier(n_estimators=300, max_depth=3, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)),
        ])
    raise ValueError(model_name)


def clean_feature_matrix(data: pd.DataFrame, feature_cols: list[str]) -> pd.DataFrame:
    X = data[feature_cols].apply(pd.to_numeric, errors='coerce')
    return X.replace([np.inf, -np.inf], np.nan)


def get_probability(model, X: pd.DataFrame) -> np.ndarray:
    proba = model.predict_proba(X)
    classes = list(model.classes_) if hasattr(model, 'classes_') else list(model[-1].classes_)
    return proba[:, classes.index(1)]


def metric_row(y_true, y_prob, threshold: float) -> dict:
    y_pred = (np.asarray(y_prob) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    try:
        auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = np.nan
    return {
        'threshold': threshold,
        'rows': int(len(y_true)),
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'roc_auc': auc,
        'precision_target': precision_score(y_true, y_pred, zero_division=0),
        'recall_target': recall_score(y_true, y_pred, zero_division=0),
        'f1_target': f1_score(y_true, y_pred, zero_division=0),
        'normal_fpr': fp / (fp + tn) if (fp + tn) else np.nan,
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
    }


## Evaluation

In [7]:

prediction_rows = []
fold_metric_rows = []
for dataset_name, data in DATASETS.items():
    y = data['binary_label'].astype(int).to_numpy()
    groups = data['substation_id'].astype(int).to_numpy()
    splitter = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    splits = list(splitter.split(data, y, groups))
    for feature_set, feature_cols in FEATURE_COLUMNS.items():
        X = clean_feature_matrix(data, feature_cols)
        for fold, (train_idx, test_idx) in enumerate(splits, start=1):
            train_groups = set(groups[train_idx]); test_groups = set(groups[test_idx])
            assert not (train_groups & test_groups), f'group overlap {dataset_name} {feature_set} fold {fold}'
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]
            test_data = data.iloc[test_idx].copy()
            for model_name in MODEL_NAMES:
                model = build_model(model_name)
                model.fit(X_train, y_train)
                y_prob = get_probability(model, X_test)
                pred_frame = test_data[['source_id','final_class','target_class','policy_variant','source_event_id','fault_event_id','disturbance_row_id','substation_id','window_policy','window_start','window_end','coverage_rate','overlap_disturbance_count','hard_normal_tag']].copy()
                pred_frame['dataset'] = dataset_name
                pred_frame['feature_set'] = feature_set
                pred_frame['feature_count'] = len(feature_cols)
                pred_frame['model'] = model_name
                pred_frame['fold'] = fold
                pred_frame['y_true'] = y_test
                pred_frame['probability_target'] = y_prob
                for threshold in THRESHOLDS:
                    pred_frame[f'prediction_t{str(threshold).replace(".", "_")}'] = (y_prob >= threshold).astype(int)
                    row = metric_row(y_test, y_prob, threshold)
                    row.update({
                        'dataset': dataset_name,
                        'target_class': data['target_class'].iloc[0],
                        'policy_variant': data['policy_variant'].iloc[0],
                        'feature_set': feature_set,
                        'feature_count': len(feature_cols),
                        'model': model_name,
                        'fold': fold,
                        'metric_scope': 'fold',
                        'train_rows': int(len(train_idx)),
                        'test_rows': int(len(test_idx)),
                        'train_groups': '|'.join(map(str, sorted(train_groups))),
                        'test_groups': '|'.join(map(str, sorted(test_groups))),
                        'group_overlap_count': 0,
                    })
                    fold_metric_rows.append(row)
                prediction_rows.extend(pred_frame.to_dict('records'))

predictions = pd.DataFrame(prediction_rows)
fold_metrics = pd.DataFrame(fold_metric_rows)
predictions.to_csv(OUT_PREDICTIONS, index=False, encoding='utf-8-sig')
print('predictions', predictions.shape)
print('fold_metrics', fold_metrics.shape)


predictions (5208, 24)
fold_metrics (960, 27)


In [8]:

aggregate_rows = []
class_rows = []
confusion_rows = []
error_rows = []
for keys, group in predictions.groupby(['dataset','target_class','policy_variant','feature_set','feature_count','model'], dropna=False):
    dataset, target_class, policy_variant, feature_set, feature_count, model_name = keys
    y_true = group['y_true'].astype(int).to_numpy()
    y_prob = group['probability_target'].astype(float).to_numpy()
    for threshold in THRESHOLDS:
        pred_col = f'prediction_t{str(threshold).replace(".", "_")}'
        y_pred = group[pred_col].astype(int).to_numpy()
        row = metric_row(y_true, y_prob, threshold)
        row.update({
            'dataset': dataset, 'target_class': target_class, 'policy_variant': policy_variant,
            'feature_set': feature_set, 'feature_count': int(feature_count), 'model': model_name,
            'fold': 'all', 'metric_scope': 'aggregate', 'train_rows': np.nan, 'test_rows': int(len(group)),
            'train_groups': np.nan, 'test_groups': np.nan, 'group_overlap_count': 0,
        })
        aggregate_rows.append(row)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
        confusion_rows.append({'dataset':dataset,'target_class':target_class,'policy_variant':policy_variant,'feature_set':feature_set,'model':model_name,'threshold':threshold,'tn':int(tn),'fp':int(fp),'fn':int(fn),'tp':int(tp)})
        p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=[0,1], zero_division=0)
        for label_value, label_name, precision, recall, f1, support in zip([0,1], ['normal', str(target_class)], p, r, f, s):
            class_rows.append({'dataset':dataset,'target_class':target_class,'policy_variant':policy_variant,'feature_set':feature_set,'model':model_name,'threshold':threshold,'class_label':label_name,'class_value':label_value,'precision':precision,'recall':recall,'f1':f1,'support':int(support)})
        errors = group.loc[((group['y_true'].eq(0)) & (group[pred_col].eq(1))) | ((group['y_true'].eq(1)) & (group[pred_col].eq(0)))].copy()
        errors['threshold'] = threshold
        errors['error_type'] = np.where(errors['y_true'].eq(0), 'FP', 'FN')
        error_rows.extend(errors.to_dict('records'))

aggregate_metrics = pd.DataFrame(aggregate_rows)
threshold_metrics = pd.concat([fold_metrics, aggregate_metrics], ignore_index=True)
cv_metrics = pd.concat([fold_metrics.loc[fold_metrics['threshold'].eq(DEFAULT_THRESHOLD)].copy(), aggregate_metrics.loc[aggregate_metrics['threshold'].eq(DEFAULT_THRESHOLD)].copy()], ignore_index=True)
class_metrics = pd.DataFrame(class_rows)
confusion_matrix_audit = pd.DataFrame(confusion_rows)
error_audit = pd.DataFrame(error_rows)

threshold_metrics.to_csv(OUT_THRESHOLD_METRICS, index=False, encoding='utf-8-sig')
cv_metrics.to_csv(OUT_CV_METRICS, index=False, encoding='utf-8-sig')
class_metrics.to_csv(OUT_CLASS_METRICS, index=False, encoding='utf-8-sig')
confusion_matrix_audit.to_csv(OUT_CONFUSION, index=False, encoding='utf-8-sig')
error_audit.to_csv(OUT_ERROR_AUDIT, index=False, encoding='utf-8-sig')
aggregate_metrics.sort_values(['threshold','balanced_accuracy'], ascending=[True,False]).head(20)


,threshold,rows,accuracy,balanced_accuracy,macro_f1,roc_auc,precision_target,recall_target,f1_target,normal_fpr,...,feature_set,feature_count,model,fold,metric_scope,train_rows,test_rows,train_groups,test_groups,group_overlap_count
69,0.4,82,0.853659,0.861398,0.852871,0.874164,0.926829,0.808511,0.863636,0.085714,...,expanded154,154,random_forest_balanced_depth3,all,aggregate,NaN,82,NaN,NaN,0
63,0.4,82,0.804878,0.826140,0.804762,0.883891,0.969697,0.680851,0.800000,0.028571,...,expanded154,154,extra_trees_balanced_depth3,all,aggregate,NaN,82,NaN,NaN,0
102,0.4,97,0.835052,0.814977,0.818861,0.861751,0.859375,0.887097,0.873016,0.257143,...,compact13,13,logistic_balanced,all,aggregate,NaN,97,NaN,NaN,0
66,0.4,82,0.804878,0.800608,0.800608,0.885106,0.829787,0.829787,0.829787,0.228571,...,expanded154,154,logistic_balanced,all,aggregate,NaN,82,NaN,NaN,0
78,0.4,90,0.800000,0.784416,0.787290,0.845714,0.824561,0.854545,0.839286,0.285714,...,compact13,13,logistic_balanced,all,aggregate,NaN,90,NaN,NaN,0
15,0.4,81,0.765432,0.783230,0.765289,0.846584,0.909091,0.652174,0.759494,0.085714,...,expanded154,154,extra_trees_balanced_depth3,all,aggregate,NaN,81,NaN,NaN,0
105,0.4,97,0.824742,0.782028,0.796746,0.863594,0.816901,0.935484,0.872180,0.371429,...,compact13,13,random_forest_balanced_depth3,all,aggregate,NaN,97,NaN,NaN,0
81,0.4,90,0.811111,0.777922,0.789227,0.881039,0.796875,0.927273,0.857143,0.371429,...,compact13,13,random_forest_balanced_depth3,all,aggregate,NaN,90,NaN,NaN,0
21,0.4,81,0.765432,0.766149,0.763121,0.852174,0.813953,0.760870,0.786517,0.228571,...,expanded154,154,random_forest_balanced_depth3,all,aggregate,NaN,81,NaN,NaN,0
18,0.4,81,0.728395,0.730124,0.726351,0.821739,0.785714,0.717391,0.750000,0.257143,...,expanded154,154,logistic_balanced,all,aggregate,NaN,81,NaN,NaN,0


## Decision And Quality Audit

In [9]:

dummy_lookup = aggregate_metrics.loc[aggregate_metrics['model'].eq('dummy_most_frequent'), ['dataset','feature_set','threshold','balanced_accuracy']].rename(columns={'balanced_accuracy':'dummy_balanced_accuracy'})
decision_matrix = aggregate_metrics.merge(dummy_lookup, on=['dataset','feature_set','threshold'], how='left')
decision_matrix['balanced_accuracy_lift_vs_dummy'] = decision_matrix['balanced_accuracy'] - decision_matrix['dummy_balanced_accuracy']
decision_matrix['passes_target_recall'] = decision_matrix['recall_target'] >= 0.80
decision_matrix['passes_normal_fpr'] = decision_matrix['normal_fpr'] <= 0.25
decision_matrix['passes_dummy_lift'] = decision_matrix['balanced_accuracy_lift_vs_dummy'] >= 0.15
decision_matrix['passes_group_overlap'] = decision_matrix['group_overlap_count'].fillna(0).eq(0)
decision_matrix['passes_gate_criteria'] = decision_matrix['passes_target_recall'] & decision_matrix['passes_normal_fpr'] & decision_matrix['passes_dummy_lift'] & decision_matrix['passes_group_overlap'] & ~decision_matrix['model'].eq('dummy_most_frequent')
decision_matrix['decision_candidate'] = np.where(decision_matrix['passes_gate_criteria'], 'one_vs_normal_gate_candidate_locked', 'one_vs_normal_gate_iteration_needed')

def target_decision(group: pd.DataFrame) -> str:
    default = group.loc[group['threshold'].eq(DEFAULT_THRESHOLD) & ~group['model'].eq('dummy_most_frequent')]
    return 'one_vs_normal_gate_candidate_locked' if default['passes_gate_criteria'].any() else 'one_vs_normal_gate_iteration_needed'

target_decisions = decision_matrix.groupby('target_class').apply(target_decision).rename('target_decision_at_t0_5').reset_index()
decision_matrix = decision_matrix.merge(target_decisions, on='target_class', how='left')
decision_matrix.to_csv(OUT_DECISION_MATRIX, index=False, encoding='utf-8-sig')

# Independent recheck from saved prediction CSV.
recheck_predictions = pd.read_csv(OUT_PREDICTIONS)
recheck_rows = []
for _, row in decision_matrix.loc[decision_matrix['metric_scope'].eq('aggregate')].iterrows():
    pred_col = f"prediction_t{str(row['threshold']).replace('.', '_')}"
    subset = recheck_predictions.loc[
        recheck_predictions['dataset'].eq(row['dataset']) &
        recheck_predictions['feature_set'].eq(row['feature_set']) &
        recheck_predictions['model'].eq(row['model'])
    ]
    y_true = subset['y_true'].astype(int).to_numpy()
    y_pred = subset[pred_col].astype(int).to_numpy()
    recomputed_ba = balanced_accuracy_score(y_true, y_pred)
    recomputed_macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    recheck_rows.append({
        'check_name': 'metric_recompute', 'dataset': row['dataset'], 'feature_set': row['feature_set'], 'model': row['model'], 'threshold': row['threshold'],
        'reported_balanced_accuracy': row['balanced_accuracy'], 'recomputed_balanced_accuracy': recomputed_ba,
        'reported_macro_f1': row['macro_f1'], 'recomputed_macro_f1': recomputed_macro_f1,
        'pass': bool(np.isclose(row['balanced_accuracy'], recomputed_ba) and np.isclose(row['macro_f1'], recomputed_macro_f1)),
        'note': 'independent recompute from saved predictions',
    })
source_status = subprocess.run(['git','status','--short','--','05_데이터셋/PreDist'], capture_output=True, text=True, encoding='utf-8', errors='ignore').stdout.strip()
recheck_rows.append({'check_name':'source_zip_exists','dataset':'source','feature_set':'NA','model':'NA','threshold':np.nan,'reported_balanced_accuracy':np.nan,'recomputed_balanced_accuracy':np.nan,'reported_macro_f1':np.nan,'recomputed_macro_f1':np.nan,'pass':ZIP_PATH.exists(),'note':'original zip exists'})
recheck_rows.append({'check_name':'source_metadata_clean','dataset':'source','feature_set':'NA','model':'NA','threshold':np.nan,'reported_balanced_accuracy':np.nan,'recomputed_balanced_accuracy':np.nan,'reported_macro_f1':np.nan,'recomputed_macro_f1':np.nan,'pass':source_status == '', 'note':'git status clean under source data directory'})
recheck_rows.append({'check_name':'special_event_audit_present','dataset':'audit','feature_set':'NA','model':'NA','threshold':np.nan,'reported_balanced_accuracy':np.nan,'recomputed_balanced_accuracy':np.nan,'reported_macro_f1':np.nan,'recomputed_macro_f1':np.nan,'pass':len(special_audit)==8 and special_audit['present_in_audit'].all(), 'note':'special fault and hard normal events retained'})
quality_audit = pd.DataFrame(recheck_rows)
quality_audit.to_csv(OUT_QUALITY_AUDIT, index=False, encoding='utf-8-sig')
print('decision rows', decision_matrix.shape)
print('quality pass', quality_audit['pass'].all())
decision_matrix.loc[decision_matrix['threshold'].eq(DEFAULT_THRESHOLD) & ~decision_matrix['model'].eq('dummy_most_frequent')].sort_values(['passes_gate_criteria','balanced_accuracy'], ascending=[False,False]).head(20)


decision rows (192, 36)
quality pass True


,threshold,rows,accuracy,balanced_accuracy,macro_f1,roc_auc,precision_target,recall_target,f1_target,normal_fpr,...,group_overlap_count,dummy_balanced_accuracy,balanced_accuracy_lift_vs_dummy,passes_target_recall,passes_normal_fpr,passes_dummy_lift,passes_group_overlap,passes_gate_criteria,decision_candidate,target_decision_at_t0_5
82,0.5,90,0.855556,0.845455,0.847239,0.881039,0.875000,0.890909,0.882883,0.200000,...,0,0.500000,0.345455,True,True,True,True,True,one_vs_normal_gate_candidate_locked,one_vs_normal_gate_candidate_locked
106,0.5,97,0.845361,0.829263,0.831304,0.863594,0.873016,0.887097,0.880000,0.228571,...,0,0.500000,0.329263,True,True,True,True,True,one_vs_normal_gate_candidate_locked,one_vs_normal_gate_candidate_locked
70,0.5,82,0.804878,0.826140,0.804762,0.874164,0.969697,0.680851,0.800000,0.028571,...,0,0.500000,0.326140,False,True,True,True,False,one_vs_normal_gate_iteration_needed,one_vs_normal_gate_iteration_needed
76,0.5,90,0.811111,0.824675,0.808247,0.904935,0.913043,0.763636,0.831683,0.114286,...,0,0.500000,0.324675,False,True,True,True,False,one_vs_normal_gate_iteration_needed,one_vs_normal_gate_candidate_locked
22,0.5,81,0.802469,0.822671,0.802198,0.852174,0.968750,0.673913,0.794872,0.028571,...,0,0.500000,0.322671,False,True,True,True,False,one_vs_normal_gate_iteration_needed,one_vs_normal_gate_iteration_needed
100,0.5,97,0.793814,0.813825,0.788763,0.898618,0.920000,0.741935,0.821429,0.114286,...,0,0.500000,0.313825,False,True,True,True,False,one_vs_normal_gate_iteration_needed,one_vs_normal_gate_candidate_locked
16,0.5,81,0.777778,0.800932,0.776928,0.846584,0.966667,0.630435,0.763158,0.028571,...,0,0.500000,0.300932,False,True,True,True,False,one_vs_normal_gate_iteration_needed,one_vs_normal_gate_iteration_needed
103,0.5,97,0.793814,0.795161,0.783675,0.861751,0.875000,0.790323,0.830508,0.200000,...,0,0.500000,0.295161,False,True,True,True,False,one_vs_normal_gate_iteration_needed,one_vs_normal_gate_candidate_locked
64,0.5,82,0.768293,0.794225,0.767428,0.883891,0.966667,0.617021,0.753247,0.028571,...,0,0.500000,0.294225,False,True,True,True,False,one_vs_normal_gate_iteration_needed,one_vs_normal_gate_iteration_needed
67,0.5,82,0.792683,0.793617,0.790155,0.885106,0.840909,0.787234,0.813187,0.200000,...,0,0.500000,0.293617,False,True,True,True,False,one_vs_normal_gate_iteration_needed,one_vs_normal_gate_iteration_needed


## Validation And Report

In [10]:

required_outputs = [OUT_DATASET_SUMMARY, OUT_CV_METRICS, OUT_PREDICTIONS, OUT_THRESHOLD_METRICS, OUT_CLASS_METRICS, OUT_CONFUSION, OUT_ERROR_AUDIT, OUT_DECISION_MATRIX, OUT_QUALITY_AUDIT, OUT_SPECIAL_EVENT_AUDIT]
for path in required_outputs:
    assert path.exists(), path
assert dataset_summary.groupby('target_class')['normal_rows'].min().eq(35).all()
assert threshold_metrics['group_overlap_count'].fillna(0).max() == 0
assert quality_audit['pass'].all()
for path in required_outputs:
    text = path.read_text(encoding='utf-8-sig', errors='ignore').lower()
    assert ('manufacturer' + '_2') not in text
    assert ('manufacturer' + ' ' + '2') not in text
print('validation checks passed')

best_default = decision_matrix.loc[decision_matrix['threshold'].eq(DEFAULT_THRESHOLD) & ~decision_matrix['model'].eq('dummy_most_frequent')].sort_values(['passes_gate_criteria','balanced_accuracy'], ascending=[False,False])
best_by_target = best_default.groupby('target_class').head(1).copy()
locked_targets = best_default.loc[best_default['passes_gate_criteria'], 'target_class'].drop_duplicates().tolist()
overall = 'partial_one_vs_normal_gate_candidate' if locked_targets else 'one_vs_normal_gate_iteration_needed'

def fmt(value):
    return 'NA' if pd.isna(value) else f'{float(value):.4f}'

def md_table(df: pd.DataFrame) -> str:
    table = df.copy()
    for col in table.columns:
        table[col] = table[col].map(lambda x: '' if pd.isna(x) else str(x))
    return '\n'.join(['| ' + ' | '.join(table.columns) + ' |', '| ' + ' | '.join(['---'] * len(table.columns)) + ' |'] + ['| ' + ' | '.join(row) + ' |' for row in table.astype(str).values.tolist()])

lines = []
lines.append('# M1 Event Type One-vs-Normal Gate 검증 보고서')
lines.append('')
lines.append('## 결론')
lines.append(f'- 최종 판단: `{overall}`')
if locked_targets:
    lines.append(f'- 기준을 만족한 target: `{", ".join(locked_targets)}`')
else:
    lines.append('- threshold 0.5 기준에서 fault/task/activity 중 gate 잠금 조건을 완전히 만족한 target은 없다.')
lines.append('- `normal vs event` 단일 gate보다 class별 gate가 어느 target에서 가능한지 확인하는 단계이며, 4분류 확정은 아직 보류한다.')
lines.append('')
lines.append('## 근거')
show_cols = ['target_class','dataset','feature_set','model','balanced_accuracy','macro_f1','balanced_accuracy_lift_vs_dummy','recall_target','normal_fpr','fp','fn','passes_gate_criteria','target_decision_at_t0_5']
lines.append(md_table(best_by_target[show_cols]))
lines.append('')
lines.append('## Class별 Sample 수')
lines.append(md_table(dataset_summary[['dataset','target_class','policy_variant','normal_rows','target_rows','target_overlap_rows','coverage_min','coverage_median']]))
lines.append('')
lines.append('## Class별 Precision/Recall/F1')
class_show = class_metrics.loc[class_metrics['threshold'].eq(DEFAULT_THRESHOLD)].merge(best_by_target[['target_class','dataset','feature_set','model']], on=['target_class','dataset','feature_set','model'], how='inner')
lines.append(md_table(class_show[['target_class','dataset','feature_set','model','class_label','precision','recall','f1','support']]))
lines.append('')
lines.append('## Confusion Matrix')
conf_show = confusion_matrix_audit.loc[confusion_matrix_audit['threshold'].eq(DEFAULT_THRESHOLD)].merge(best_by_target[['target_class','dataset','feature_set','model']], on=['target_class','dataset','feature_set','model'], how='inner')
lines.append(md_table(conf_show[['target_class','dataset','feature_set','model','tn','fp','fn','tp']]))
lines.append('')
lines.append('## 품질 검증')
lines.append('- 생성 CSV를 다시 읽어 balanced accuracy와 macro F1을 독립 재계산했고, 저장 metric과 일치하는지 확인했다.')
lines.append('- 모든 계산은 16번 M1 taxonomy/window audit 산출물 기준으로 수행했다.')
lines.append('- 비대상 제조사 문자열/경로/계산 결과는 새 산출물에 포함하지 않았다.')
lines.append('- 원본 ZIP 존재와 `05_데이터셋/PreDist` 하위 git status clean을 확인했다.')
lines.append('- Event 20/34/69/67 및 Event 19/68/35/48 처리 기준은 special event audit으로 유지했다.')
lines.append(md_table(quality_audit.groupby('check_name').agg(pass_all=('pass','all'), rows=('check_name','count')).reset_index()))
lines.append('')
lines.append('## 한계')
lines.append('- normal은 35건으로 고정되어 있어 normal FPR 추정이 fold 구성에 민감하다.')
lines.append('- event type별 샘플 수와 window 정책이 서로 다르기 때문에, one-vs-normal 결과를 바로 4분류 성능으로 해석하면 안 된다.')
lines.append('- threshold 0.5 기준을 우선했으며, 운영 threshold 튜닝은 아직 하지 않았다.')
lines.append('')
lines.append('## 다음 작업 순서')
lines.append('1. 기준을 만족한 target이 있으면 해당 class gate를 후보로 잠근다.')
lines.append('2. 기준 미달 target은 window 정책 또는 normal 후보 보강을 먼저 재검토한다.')
lines.append('3. class별 gate가 2개 이상 안정화된 뒤에만 `fault/task/activity` 내부 다중분류를 다시 검토한다.')
OUT_REPORT.write_text('\n'.join(lines) + '\n', encoding='utf-8')
print(OUT_REPORT)


validation checks passed
C:\3rd_Project\HeatGridAgent\07_데이터산출물\19_M1_event_type_one_vs_normal_gate_validation_보고서.md


## Takeaways

보고서의 결론, 근거, 한계, 다음 작업 순서에 따라 다음 실험을 정한다.